# TaxGPT — Episode 4: Causal Self-Attention, From Scratch

Companion notebook to blog post *"Self-Attention Explained From Scratch: Q, K, V and Scaled Dot-Product Attention (TaxGPT Episode 4)"*.

> **Same caveat as the blog post:** the 12-head, 768-dim configuration here is TaxGPT's current architecture spec — verify against the live `TransformerBlock` / attention class before treating it as final.

Reference: Sebastian Raschka, *Build a Large Language Model (From Scratch)*, Ch. 3. Andrej Karpathy, *Let's Build GPT: from scratch, in code, spelled out* (nanoGPT).

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

## 1. A single causal self-attention head, built from the raw math

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + \text{mask}\right) V$$

In [2]:
class CausalSelfAttentionHead(nn.Module):
    def __init__(self, emb_dim, head_dim, context_len):
        super().__init__()
        self.W_q = nn.Linear(emb_dim, head_dim, bias=False)
        self.W_k = nn.Linear(emb_dim, head_dim, bias=False)
        self.W_v = nn.Linear(emb_dim, head_dim, bias=False)
        self.head_dim = head_dim
        # causal mask: position i may only attend to positions <= i
        self.register_buffer("mask", torch.tril(torch.ones(context_len, context_len)))

    def forward(self, x):
        B, T, C = x.shape  # batch, seq_len, emb_dim
        Q = self.W_q(x)    # (B, T, head_dim)
        K = self.W_k(x)
        V = self.W_v(x)

        # 1. raw scores
        scores = Q @ K.transpose(-2, -1)                # (B, T, T)
        # 2. scale
        scores = scores / math.sqrt(self.head_dim)
        # 3. causal mask -> -inf on future positions
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
        # 4. softmax -> probability distribution per row
        attn_weights = F.softmax(scores, dim=-1)
        # 5. weighted sum of values
        out = attn_weights @ V                           # (B, T, head_dim)
        return out, attn_weights

EMB_DIM = 768
HEAD_DIM = 64     # 768 / 12 heads
CONTEXT_LEN = 1024

head = CausalSelfAttentionHead(EMB_DIM, HEAD_DIM, CONTEXT_LEN)
x = torch.randn(1, 6, EMB_DIM)  # pretend: batch=1, 6-token sequence, already embedded (Episode 3)
out, attn_weights = head(x)

print("input x shape:", x.shape)
print("output shape:", out.shape)
print("attn_weights shape:", attn_weights.shape)

input x shape: torch.Size([1, 6, 768])
output shape: torch.Size([1, 6, 64])
attn_weights shape: torch.Size([1, 6, 6])


## 2. Confirming causality: the mask actually zeroes out future attention

Row *i* of the attention matrix should have zero probability mass on any column *j > i*.

In [3]:
torch.set_printoptions(precision=3, sci_mode=False)
print("attention weights (rows=query position, cols=key position):")
print(attn_weights[0])
print()

# check: is everything strictly above the diagonal zero?
upper_triangle = torch.triu(attn_weights[0], diagonal=1)
print("max attention weight placed on a *future* token (should be 0.0):", upper_triangle.max().item())

attention weights (rows=query position, cols=key position):
tensor([[1.000, 0.000, 0.000, 0.000, 0.000, 0.000],
        [0.630, 0.370, 0.000, 0.000, 0.000, 0.000],
        [0.333, 0.249, 0.419, 0.000, 0.000, 0.000],
        [0.289, 0.108, 0.226, 0.378, 0.000, 0.000],
        [0.267, 0.211, 0.279, 0.080, 0.163, 0.000],
        [0.150, 0.207, 0.140, 0.189, 0.169, 0.145]], grad_fn=<SelectBackward0>)

max attention weight placed on a *future* token (should be 0.0): 0.0


## 3. Multi-head attention — 12 heads run in parallel, then merged

This is the actual TaxGPT configuration: 12 heads × 64 dims = 768 (the full embedding dimension).

In [4]:
class CausalMultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, n_heads, context_len):
        super().__init__()
        assert emb_dim % n_heads == 0
        head_dim = emb_dim // n_heads
        self.heads = nn.ModuleList([
            CausalSelfAttentionHead(emb_dim, head_dim, context_len) for _ in range(n_heads)
        ])
        self.out_proj = nn.Linear(emb_dim, emb_dim)

    def forward(self, x):
        head_outputs = [h(x)[0] for h in self.heads]     # each: (B, T, head_dim)
        concatenated = torch.cat(head_outputs, dim=-1)   # (B, T, emb_dim)
        return self.out_proj(concatenated)

N_HEADS = 12
mha = CausalMultiHeadAttention(EMB_DIM, N_HEADS, CONTEXT_LEN)
mha_out = mha(x)

print(f"{N_HEADS} heads x {EMB_DIM // N_HEADS} head_dim = {N_HEADS * (EMB_DIM // N_HEADS)} (should equal EMB_DIM={EMB_DIM})")
print("multi-head output shape:", mha_out.shape, " -> same shape as input, ready for the next transformer block")

n_params = sum(p.numel() for p in mha.parameters())
print(f"parameters in this one multi-head attention block: {n_params:,}")

12 heads x 64 head_dim = 768 (should equal EMB_DIM=768)
multi-head output shape: torch.Size([1, 6, 768])  -> same shape as input, ready for the next transformer block
parameters in this one multi-head attention block: 2,360,064


## 4. Why this matters for GST text — a concrete attention pattern

We can't easily *train* a demo on 10M tokens in this notebook, but here's the qualitative claim from the blog post made visible: with a long-range dependency like *"Input Tax Credit ... shall not be available"*, a trained model's attention weights for the token after "available" should place meaningful weight back on "Credit" — several positions earlier — not just on the immediately preceding token. That long-range weighting is exactly what the softmax row over all *earlier* positions (step 4 above) makes possible, and it's what RNN-based sequence models struggled to do reliably.

## Takeaway

Self-attention lets every token's representation get updated based on the other tokens that matter for it — causally, so the model never cheats by looking ahead. On its own it's not enough to train a deep, stable network: that's where layer norm and residual connections come in, covered in the next episode.